# Re-score an existing AF checkpoint under the honest protocol

Evaluation only. No training. About 15 minutes.

Use this for any checkpoint that was trained before the protocol changed:

| Set `TS_LENGTH` | Set `CKPT_NAME` to match | Purpose |
|---|---|---|
| 1 | your TS=1 checkpoint | makes TS=1 comparable to TS=2 |
| 2 | your TS=2 seed-42 checkpoint | completes the AF seed study |

What it does, identical to the merged training notebook:

1. Sweeps the decision threshold on **validation** (0.05 to 0.90)
2. Applies that threshold to the test set -- no test-set peeking
3. Evaluates all 17 official AF test fires
4. Reports aggregates over all evaluated fires, the 15 with labels, and the 14
   excluding `double_creek_fire`
5. Prints the test-swept optimum too, labelled as an oracle bound only

Attach: the TS-SatFire dataset and the dataset holding your checkpoints.


In [1]:
# --- Cell 1: Imports ---
import os, gc, sys, time, glob, random, warnings, json, math
from datetime import datetime
from collections import OrderedDict

PIPELINE_START = time.time()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score
from tqdm.auto import tqdm

try:
    import rasterio
except ImportError:
    os.system("pip install rasterio --quiet")
    import rasterio

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Device:  {DEVICE}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} -- {p.total_memory/1e9:.1f} GB")
print(f"Setup: {time.time()-PIPELINE_START:.1f}s")




Python:  3.12.13
PyTorch: 2.10.0+cu128
CUDA:    12.8
Device:  cuda
  GPU 0: Tesla T4 -- 15.6 GB
  GPU 1: Tesla T4 -- 15.6 GB
Setup: 7.1s


In [2]:
# ----------------------------------------------------------------------
# SET THESE TWO
# ----------------------------------------------------------------------
TS_LENGTH = 1            # 1 for the TS=1 checkpoint, 2 for the TS=2 one
CKPT_NAME = "best_v6.pt"      # e.g. "best_af_ts1.pt". None -> pick from the list printed below
SEED_LABEL = 42          # only used to name the output files
# ----------------------------------------------------------------------


class Config:
    DATA_ROOT = ""
    for _p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire/ts-satfire",
               "/kaggle/input/ts-satfire"]:
        if os.path.isdir(_p):
            DATA_ROOT = _p
            break
    OUTPUT_DIR = "/kaggle/working"

    IMAGE_SIZE = 256
    N_CHANNELS = 8
    MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                     294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                    24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    BATCH_SIZE = 8
    NUM_WORKERS = 2
    USE_AMP = True
    ENCODER_CHANNELS = [64, 128, 256, 512]
    DROPOUT = 0.1
    SE_REDUCTION = 8
    MIN_FIRE_PX = 10
    MAX_NEG_RATIO = 2
    SEED = 42

    VAL_IDS = ["20568194", "20701026", "20562846", "20700973", "24462610",
               "24462788", "24462753", "24103571", "21998313", "21751303",
               "22141596", "21999381", "22712904"]
    NO_LABEL_IDS = [
        "20777207", "20777386", "21693566", "21751309",
        "21889672", "21889683", "21889697", "21889719",
        "21889734", "21889754", "21997775", "22712973",
        "22713339", "23860939", "23860978", "23861018",
        "23861131", "24332700", "22712904",
    ]

cfg = Config()
cfg.TS_LENGTH = TS_LENGTH
assert cfg.DATA_ROOT, "TS-SatFire dataset not found"
random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)

SFX = f"_ts{TS_LENGTH}_s{SEED_LABEL}"
print(f"DATA_ROOT: {cfg.DATA_ROOT}")
print(f"TS_LENGTH: {TS_LENGTH} | output suffix '{SFX}'")

# ---- locate checkpoints ----
ds_root = os.path.abspath(cfg.DATA_ROOT)
cands = []
for base in ["/kaggle/input", "/kaggle/working"]:
    if not os.path.isdir(base):
        continue
    for root, dirs, files in os.walk(base):
        if os.path.abspath(root).startswith(ds_root):
            dirs.clear(); continue
        for f in files:
            if f.endswith((".pt", ".pth", ".ckpt")):
                p = os.path.join(root, f)
                cands.append((p, os.path.getsize(p) / 1e6))
cands.sort(key=lambda x: -x[1])

print("\nCheckpoints found:")
for p, s in cands:
    print(f"  {s:8.1f} MB  {p}")
assert cands, "No checkpoints found. Attach the dataset holding them."

if CKPT_NAME:
    hit = [p for p, _ in cands if os.path.basename(p) == CKPT_NAME]
    assert hit, f"{CKPT_NAME} not found among the checkpoints above"
    CKPT = hit[0]
else:
    CKPT = cands[0][0]
print(f"\nUsing: {CKPT}")
print("Set CKPT_NAME above if this is the wrong one.")


DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire
TS_LENGTH: 1 | output suffix '_ts1_s42'

Checkpoints found:
     131.6 MB  /kaggle/input/datasets/panosxoblas/spase-unet3d-checkpoints/best_v6.pt

Using: /kaggle/input/datasets/panosxoblas/spase-unet3d-checkpoints/best_v6.pt
Set CKPT_NAME above if this is the wrong one.


In [3]:
def load_frame(fire_dir, day_path, return_label=False):
    """Load 8-channel frame + optional AF label."""
    with rasterio.open(day_path) as src:
        day_arr = src.read().astype(np.float32)
    day_bands = day_arr[:6]
    label = None
    if return_label and day_arr.shape[0] >= 7:
        b7 = day_arr[6]
        if np.isnan(b7).sum() < b7.size:  # not all NaN
            label = (b7 >= 7).astype(np.float32)

    night_dir = os.path.join(fire_dir, "VIIRS_Night")
    night_path = os.path.join(night_dir,
        os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(night_path):
        with rasterio.open(night_path) as src:
            na = src.read().astype(np.float32)
        nb = na[:2] if na.shape[0] >= 2 else np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    else:
        nb = np.zeros((2, *day_bands.shape[1:]), dtype=np.float32)
    frame = np.concatenate([day_bands, nb], axis=0)
    return (frame, label) if return_label else frame


def check_day_has_label(day_path):
    """Quick check if band 7 has any non-NaN values."""
    with rasterio.open(day_path) as src:
        if src.count < 7:
            return False
        b7 = src.read(7).astype(np.float32)
        return np.isnan(b7).sum() < b7.size


# Build clean train/val splits
all_ids = sorted(os.listdir(cfg.DATA_ROOT))
numeric_ids = [d for d in all_ids if d.isdigit()]

# Exclude fires with no labels
clean_train_ids = [d for d in numeric_ids
                   if d not in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]
clean_val_ids = [d for d in numeric_ids
                 if d in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]

train_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_train_ids]
val_fires = [os.path.join(cfg.DATA_ROOT, d) for d in clean_val_ids]

print(f"Original:  {len(numeric_ids)} fires")
print(f"Excluded:  {len(cfg.NO_LABEL_IDS)} fires (zero labels)")
print(f"Clean train: {len(train_fires)} fires")
print(f"Clean val:   {len(val_fires)} fires")
print(f"Removed from train: {len(numeric_ids) - len(cfg.VAL_IDS) - len(train_fires)} fires")
print(f"Removed from val:   {len(cfg.VAL_IDS) - len(val_fires)} fires")




class AFValDataset(Dataset):
    """Validation windows only, for the threshold sweep."""

    def __init__(self, fire_dirs, cfg):
        self.T, self.ps = cfg.TS_LENGTH, cfg.IMAGE_SIZE
        self.means, self.stds = cfg.MEAN, cfg.STD
        self.samples = []
        rng = random.Random(cfg.SEED)
        for fd in fire_dirs:
            files = sorted(glob.glob(os.path.join(fd, "VIIRS_Day", "*.tif")))
            if len(files) < self.T:
                continue
            try:
                with rasterio.open(files[0]) as src:
                    if src.count < 7:
                        continue
                    H, W = src.height, src.width
            except Exception:
                continue
            if H < self.ps or W < self.ps:
                continue
            start = 0
            while start + self.T <= len(files):
                last = files[start + self.T - 1]
                lbl = None
                try:
                    with rasterio.open(last) as src:
                        if src.count >= 7:
                            b7 = src.read(7).astype(np.float32)
                            if np.isnan(b7).sum() < b7.size:
                                lbl = (b7 >= 7).astype(np.float32)
                except Exception:
                    pass
                if lbl is not None:
                    r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
                    px = int(lbl[r0:r0 + self.ps, c0:c0 + self.ps].sum())
                    if px >= cfg.MIN_FIRE_PX or rng.random() < 1.0 / (cfg.MAX_NEG_RATIO + 1):
                        self.samples.append({"fd": fd, "files": files,
                                             "start": start, "H": H, "W": W})
                start += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fd, H, W = s["fd"], s["H"], s["W"]
        win = s["files"][s["start"]:s["start"] + self.T]
        frames, label = [], None
        for t, dp in enumerate(win):
            if t == len(win) - 1:
                fr, label = load_frame(fd, dp, return_label=True)
            else:
                fr = load_frame(fd, dp)
            frames.append(fr[:, :H, :W])
        if label is None:
            label = np.zeros((H, W), np.float32)
        label = label[:H, :W]
        stack = np.stack(frames, axis=0)
        stack = (stack - self.means[None, :, None, None]) / (self.stds[None, :, None, None] + 1e-8)
        stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)
        r0, c0 = (H - self.ps) // 2, (W - self.ps) // 2
        stack = stack[:, :, r0:r0 + self.ps, c0:c0 + self.ps]
        label = label[r0:r0 + self.ps, c0:c0 + self.ps]
        x = torch.from_numpy(stack.transpose(1, 0, 2, 3).copy()).float()
        y = torch.from_numpy(label.copy()).long()
        return x, y


val_ids = [d for d in sorted(os.listdir(cfg.DATA_ROOT))
           if d.isdigit() and d in cfg.VAL_IDS and d not in cfg.NO_LABEL_IDS]
VAL_DS = AFValDataset([os.path.join(cfg.DATA_ROOT, d) for d in val_ids], cfg)
print(f"val fires: {len(val_ids)} | val windows: {len(VAL_DS)}")


Original:  151 fires
Excluded:  19 fires (zero labels)
Clean train: 120 fires
Clean val:   12 fires
Removed from train: 18 fires
Removed from val:   1 fires
val fires: 12 | val windows: 205


In [4]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(nn.Linear(ch, ch//r, bias=False), nn.ReLU(True),
                                nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlock3D(nn.Module):
    def __init__(self, ic, oc, r=8, dr=0.1):
        super().__init__()
        self.c1 = nn.Conv3d(ic, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b1 = nn.BatchNorm3d(oc)
        self.c2 = nn.Conv3d(oc, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.b2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, r)
        self.relu = nn.ReLU(True)
        self.drop = nn.Dropout3d(dr) if dr > 0 else nn.Identity()
        self.skip = (nn.Sequential(nn.Conv3d(ic, oc, 1, bias=False),
                     nn.BatchNorm3d(oc)) if ic != oc else nn.Identity())

    def forward(self, x):
        r = self.skip(x)
        o = self.relu(self.b1(self.c1(x)))
        o = self.drop(o)
        o = self.b2(self.c2(o))
        o = self.se(o)
        return self.relu(o + r)


class SEUNet3D(nn.Module):
    def __init__(self, ic=8, nc=1, ec=(64,128,256,512), r=8, dr=0.1):
        super().__init__()
        self.e1 = ResBlock3D(ic, ec[0], r, dr)
        self.e2 = ResBlock3D(ec[0], ec[1], r, dr)
        self.e3 = ResBlock3D(ec[1], ec[2], r, dr)
        self.e4 = ResBlock3D(ec[2], ec[3], r, dr)
        self.pool = nn.MaxPool3d((1,2,2), stride=(1,2,2))
        self.bot = ResBlock3D(ec[3], ec[3]*2, r, dr)

        self.u4 = nn.ConvTranspose3d(ec[3]*2, ec[3], (1,2,2), stride=(1,2,2))
        self.d4 = ResBlock3D(ec[3]*2, ec[3], r, dr)
        self.u3 = nn.ConvTranspose3d(ec[3], ec[2], (1,2,2), stride=(1,2,2))
        self.d3 = ResBlock3D(ec[2]*2, ec[2], r, dr)
        self.u2 = nn.ConvTranspose3d(ec[2], ec[1], (1,2,2), stride=(1,2,2))
        self.d2 = ResBlock3D(ec[1]*2, ec[1], r, dr)
        self.u1 = nn.ConvTranspose3d(ec[1], ec[0], (1,2,2), stride=(1,2,2))
        self.d1 = ResBlock3D(ec[0]*2, ec[0], r, dr)

        self.final = nn.Conv3d(ec[0], nc, 1)
        self.ds3 = nn.Conv3d(ec[2], nc, 1)  # deep supervision

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.bot(self.pool(e4))

        d4 = self.d4(torch.cat([self.u4(b), e4], 1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))

        out = self.final(d1)
        if self.training:
            ds = F.interpolate(self.ds3(d3), size=out.shape[2:],
                               mode="trilinear", align_corners=False)
            return out, ds
        return out


class DiceFocalLoss(nn.Module):
    def __init__(self, dw=0.5, fw=0.5, gamma=2.0, alpha=0.75, dsw=0.3):
        super().__init__()
        self.dw, self.fw, self.gamma, self.alpha, self.dsw = dw, fw, gamma, alpha, dsw

    def _dice(self, p, t):
        ps = torch.sigmoid(p).reshape(-1); tf = t.reshape(-1)
        return 1 - (2*(ps*tf).sum()+1) / (ps.sum()+tf.sum()+1)

    def _focal(self, p, t):
        bce = F.binary_cross_entropy_with_logits(p, t, reduction="none")
        pt = torch.sigmoid(p)*t + (1-torch.sigmoid(p))*(1-t)
        at = self.alpha*t + (1-self.alpha)*(1-t)
        return (at * (1-pt)**self.gamma * bce).mean()

    def _loss(self, p, t):
        return self.dw*self._dice(p, t) + self.fw*self._focal(p, t)

    def forward(self, preds, target):
        main = preds[0] if isinstance(preds, tuple) else preds
        ds = preds[1] if isinstance(preds, tuple) else None
        pred_last = main[:, :, -1, :, :]
        tgt = target.unsqueeze(1).float()
        loss = self._loss(pred_last, tgt)
        if ds is not None:
            loss += self.dsw * self._loss(ds[:, :, -1, :, :], tgt)
        return loss



model = SEUNet3D(ic=cfg.N_CHANNELS, nc=1, ec=tuple(cfg.ENCODER_CHANNELS),
                 r=cfg.SE_REDUCTION, dr=cfg.DROPOUT).to(DEVICE)

ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
state = ck.get("model_state_dict", ck.get("state_dict", ck))
if any(k.startswith("module.") for k in state):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Loaded checkpoint. Params: {sum(p.numel() for p in model.parameters()):,}")
if missing or unexpected:
    print(f"  WARNING missing={len(missing)} unexpected={len(unexpected)}")
    print(f"  first missing: {missing[:3]}")
    print("  A large count here means the checkpoint does not match this architecture.")
else:
    print("  All weights matched.")
if "epoch" in ck:
    print(f"  from epoch {ck['epoch']}")


Loaded checkpoint. Params: 32,876,034
  All weights matched.
  from epoch 55


In [5]:
# Threshold sweep on VALIDATION
vl = DataLoader(VAL_DS, batch_size=cfg.BATCH_SIZE, shuffle=False,
                num_workers=cfg.NUM_WORKERS, pin_memory=True)
P, L = [], []
with torch.no_grad():
    for xb, yb in tqdm(vl, desc="val", ncols=80):
        xb = xb.to(DEVICE, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            out = model(xb)
        lg = out[0] if isinstance(out, tuple) else out
        P.append(torch.sigmoid(lg[:, 0, -1].float()).cpu().numpy().ravel())
        L.append(yb.numpy().ravel())
P = np.concatenate(P); L = np.concatenate(L) > 0.5

rows = []
for t in np.arange(0.05, 0.91, 0.02):
    p = P > t
    tp = int((p & L).sum()); fp = int((p & ~L).sum()); fn = int((~p & L).sum())
    rows.append({"thr": float(t),
                 "f1": 2 * tp / max(2 * tp + fp + fn, 1),
                 "iou": tp / max(tp + fp + fn, 1)})
sweep = pd.DataFrame(rows)
best = sweep.loc[sweep.f1.idxmax()]
VAL_THR, VAL_F1 = float(best.thr), float(best.f1)
print(f"\nValidation optimum: threshold {VAL_THR:.2f}  F1 {VAL_F1:.4f}")
print(f"F1 is within 0.002 of the optimum for thresholds "
      f"{sweep[sweep.f1 > VAL_F1 - 0.002].thr.min():.2f} to "
      f"{sweep[sweep.f1 > VAL_F1 - 0.002].thr.max():.2f}  (flat region)")


val:   0%|                                               | 0/26 [00:00<?, ?it/s]


Validation optimum: threshold 0.69  F1 0.8237
F1 is within 0.002 of the optimum for thresholds 0.45 to 0.85  (flat region)


In [6]:
# Inference utilities. load_frame is already defined above.

def prepare_window(fire_dir, day_files, t_start, ts_length, mean, std, patch_size):
    """Load a T-length window, normalize, center crop. Returns (1,C,T,H,W) tensor + label."""
    frames, label = [], None
    H = W = None
    for t in range(t_start, t_start + ts_length):
        is_last = (t == t_start + ts_length - 1)
        if is_last:
            fr, label = load_frame(fire_dir, day_files[t], return_label=True)
        else:
            fr = load_frame(fire_dir, day_files[t])
        if H is None:
            H, W = fr.shape[1], fr.shape[2]
        frames.append(fr[:, :H, :W])

    if label is not None:
        label = label[:H, :W]

    stack = np.stack(frames, axis=0)  # (T, 8, H, W)
    stack = (stack - mean[None, :, None, None]) / (std[None, :, None, None] + 1e-8)
    stack = np.nan_to_num(stack, nan=0.0, posinf=0.0, neginf=0.0)

    # Center crop
    r0 = (H - patch_size) // 2; c0 = (W - patch_size) // 2
    stack = stack[:, :, r0:r0+patch_size, c0:c0+patch_size]
    if label is not None:
        label = label[r0:r0+patch_size, c0:c0+patch_size]

    x = torch.from_numpy(stack.transpose(1, 0, 2, 3)).float().unsqueeze(0)  # (1,C,T,H,W)
    return x, label, frames[-1]  # also return raw last frame for visualization

print("Data utilities ready.")




Data utilities ready.


In [7]:
@torch.no_grad()
def evaluate_fire(fire_id, threshold=0.5):
    """Run inference on one fire, return metrics + predictions for vis."""
    fdir = os.path.join(DATA_ROOT, fire_id)
    day_files = sorted(glob.glob(os.path.join(fdir, "VIIRS_Day", "*.tif")))

    if len(day_files) < TS_LENGTH:
        return None

    tp_total = fp_total = fn_total = 0
    n_windows = 0
    n_skipped = 0
    all_probs = []
    all_labels = []
    vis_data = []  # for qualitative plots

    for t0 in range(len(day_files) - TS_LENGTH + 1):
        x, label, raw_frame = prepare_window(
            fdir, day_files, t0, TS_LENGTH, MEAN, STD, IMAGE_SIZE)

        if label is None:
            n_skipped += 1
            continue

        x = x.to(DEVICE)
        with autocast(enabled=True):
            logits = model(x)
        probs = torch.sigmoid(logits[:, 0, -1]).cpu().numpy()[0]  # (H, W)
        pred = (probs > threshold).astype(np.float32)
        lbl = label.astype(np.float32)

        tp = int(((pred == 1) & (lbl == 1)).sum())
        fp = int(((pred == 1) & (lbl == 0)).sum())
        fn = int(((pred == 0) & (lbl == 1)).sum())

        tp_total += tp; fp_total += fp; fn_total += fn
        n_windows += 1
        all_probs.append(probs.flatten())
        all_labels.append(lbl.flatten())

        # Save last window for visualization
        vis_data.append({
            "raw": raw_frame, "label": lbl, "pred": pred, "probs": probs,
            "date": os.path.basename(day_files[t0 + TS_LENGTH - 1]).replace("_VIIRS_Day.tif", "")
        })

    if n_windows == 0:
        return None

    prec = tp_total / max(tp_total + fp_total, 1)
    rec = tp_total / max(tp_total + fn_total, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-8)
    iou = tp_total / max(tp_total + fp_total + fn_total, 1)

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    return {
        "fire_id": fire_id, "n_windows": n_windows, "n_skipped": n_skipped,
        "tp": tp_total, "fp": fp_total, "fn": fn_total,
        "f1": f1, "iou": iou, "precision": prec, "recall": rec,
        "probs": all_probs, "labels": all_labels,
        "vis": vis_data,
    }



AF_TEST_ALL = [
    "elephant_hill_fire", "eagle_bluff_fire", "double_creek_fire", "sparks_lake_fire",
    "lytton_fire", "chuckegg_creek_fire", "swedish_fire", "sydney_fire",
    "thomas_fire", "tubbs_fire", "carr_fire", "camp_fire",
    "creek_fire", "blue_ridge_fire", "dixie_fire", "mosquito_fire", "calfcanyon_fire",
]
NO_LABEL_TEST = ["mosquito_fire", "calfcanyon_fire"]
PARTIAL_TEST = ["double_creek_fire"]

DATA_ROOT = cfg.DATA_ROOT
IMAGE_SIZE = cfg.IMAGE_SIZE
MEAN, STD = cfg.MEAN, cfg.STD

print(f"Evaluating {len(AF_TEST_ALL)} test fires at threshold {VAL_THR:.2f}\n")
print(f"{'Fire':<25s} {'Win':>4} {'Skip':>4} {'TP':>7} {'FP':>7} {'FN':>7} {'F1':>7} {'IoU':>7}")
print("-" * 82)

all_results, all_probs_t, all_labels_t = [], [], []
for fid in AF_TEST_ALL:
    r = evaluate_fire(fid, VAL_THR)
    if r is None:
        print(f"{fid:<25s}  no scorable windows (no labels)")
        continue
    all_results.append(r)
    all_probs_t.append(r["probs"]); all_labels_t.append(r["labels"])
    print(f"{fid:<25s} {r['n_windows']:>4} {r['n_skipped']:>4} "
          f"{r['tp']:>7} {r['fp']:>7} {r['fn']:>7} {r['f1']:>7.4f} {r['iou']:>7.4f}")


def agg(results, subset=None):
    rs = [r for r in results if subset is None or r["fire_id"] in subset]
    tp = sum(r["tp"] for r in rs); fp = sum(r["fp"] for r in rs); fn = sum(r["fn"] for r in rs)
    return {"n_fires": len(rs),
            "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "iou": tp / max(tp + fp + fn, 1),
            "macro_f1": float(np.mean([r["f1"] for r in rs])) if rs else float("nan")}


SUB15 = set(f for f in AF_TEST_ALL if f not in NO_LABEL_TEST)
SUB14 = set(f for f in SUB15 if f not in PARTIAL_TEST)
AGG = {"all_evaluated": agg(all_results),
       "labelled_15": agg(all_results, SUB15),
       "excl_double_creek_14": agg(all_results, SUB14)}

print(f"\n{'='*82}")
print(f"TS={TS_LENGTH} TEST RESULTS at the validation threshold {VAL_THR:.2f}")
print(f"{'='*82}")
print(f"{'subset':<26} {'fires':>5} {'micro F1':>10} {'micro IoU':>10} {'macro F1':>10}")
for k, a in AGG.items():
    print(f"{k:<26} {a['n_fires']:>5} {a['f1']:>10.4f} {a['iou']:>10.4f} {a['macro_f1']:>10.4f}")

# oracle bound, reported but not to be quoted
Pt = np.concatenate(all_probs_t); Lt = np.concatenate(all_labels_t) > 0.5
orows = []
for t in np.arange(0.05, 0.91, 0.02):
    p = Pt > t
    tp = int((p & Lt).sum()); fp = int((p & ~Lt).sum()); fn = int((~p & Lt).sum())
    orows.append((float(t), 2 * tp / max(2 * tp + fp + fn, 1)))
ot, of1 = max(orows, key=lambda r: r[1])
print(f"\nOracle (threshold swept on test): {ot:.2f} -> F1 {of1:.4f}")
print(f"Cost of honest selection: {AGG['labelled_15']['f1'] - of1:+.4f}")


Evaluating 17 test fires at threshold 0.69

Fire                       Win Skip      TP      FP      FN      F1     IoU
----------------------------------------------------------------------------------
elephant_hill_fire          10    0    5338    1469    1095  0.8063  0.6755
eagle_bluff_fire             5    5     515      93      81  0.8555  0.7475
double_creek_fire            3    7     236      25      59  0.8489  0.7375
sparks_lake_fire             9    1    5909    1251     804  0.8519  0.7420
lytton_fire                 10    0    1347     477     252  0.7870  0.6488
chuckegg_creek_fire          8    2   15731    2963    3861  0.8218  0.6975
swedish_fire                 9    1     839     161     118  0.8574  0.7504
sydney_fire                 10    0    8195    1900    1554  0.8259  0.7035
thomas_fire                 10    0   10632    1479    1916  0.8623  0.7580
tubbs_fire                  10    0    7554    1050    1633  0.8492  0.7379
carr_fire                   10    0  

In [8]:
pd.DataFrame([{
    "ts": TS_LENGTH, "seed": SEED_LABEL, "fire_id": r["fire_id"],
    "n_windows": r["n_windows"], "n_skipped": r["n_skipped"],
    "tp": r["tp"], "fp": r["fp"], "fn": r["fn"],
    "f1": r["f1"], "iou": r["iou"],
    "precision": r["precision"], "recall": r["recall"],
} for r in all_results]).to_csv(
    os.path.join(cfg.OUTPUT_DIR, f"af_per_fire{SFX}.csv"), index=False)

summary = {
    "model": "SE-UNet3D-AF-v6",
    "ts": TS_LENGTH,
    "seed": SEED_LABEL,
    "checkpoint": os.path.basename(CKPT),
    "n_params": sum(p.numel() for p in model.parameters()),
    "val_selected_threshold": VAL_THR,
    "val_f1_at_threshold": VAL_F1,
    "test_micro_f1_15": AGG["labelled_15"]["f1"],
    "test_micro_iou_15": AGG["labelled_15"]["iou"],
    "test_macro_f1_15": AGG["labelled_15"]["macro_f1"],
    "test_micro_f1_14_no_double_creek": AGG["excl_double_creek_14"]["f1"],
    "test_micro_f1_all_evaluated": AGG["all_evaluated"]["f1"],
    "n_fires_evaluated": AGG["all_evaluated"]["n_fires"],
    "test_oracle_threshold": float(ot),
    "test_oracle_f1": float(of1),
}
with open(os.path.join(cfg.OUTPUT_DIR, f"af_results{SFX}.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f"\nSend me af_results{SFX}.json")


{
  "model": "SE-UNet3D-AF-v6",
  "ts": 1,
  "seed": 42,
  "checkpoint": "best_v6.pt",
  "n_params": 32876034,
  "val_selected_threshold": 0.6900000000000002,
  "val_f1_at_threshold": 0.8237060766843528,
  "test_micro_f1_15": 0.8520112723097798,
  "test_micro_iou_15": 0.7421773853337794,
  "test_macro_f1_15": 0.8419179833941619,
  "test_micro_f1_14_no_double_creek": 0.8520184626457825,
  "test_micro_f1_all_evaluated": 0.8520112723097798,
  "n_fires_evaluated": 15,
  "test_oracle_threshold": 0.15000000000000002,
  "test_oracle_f1": 0.8540590661016536
}

Send me af_results_ts1_s42.json
